In [27]:
import csv

# make sure the builtin print isn’t shadowed by a string variable
try:
    del print
except NameError:
    pass

def read_csv(file_path):
    with open(file_path, mode='r', encoding='utf-8') as f:
        return list(csv.DictReader(f))

data = read_csv(r'D:\SwedishGeoHistori\SwedishGeoHistory\scraper\krig_events.csv')
# Print first 5 rows to inspect the data structure
for i, row in enumerate(data[:5]):
    print(f"Row {i+1}: {row}")

Row 1: {'year': '500', 'title': 'Krig/konflikt i Sverige 500', 'description': 'Gustav IV Adolf gav inte upp hoppet. Han lyckades, med Preussens hjälp, samla ihop en styrka om 17 500 man, delvis undermåligt tränade. Mot dessa stod den franska armén om 40 000 man. Den 13 juni 1807 började den svenska armén röra på sig, men i början av juli slöt Ryssland och Preussen fred med Frankrike. Den svenska styrkan tvingades därför dra sig tillbaka till Stralsund, varefter man snabbt backade till Rügen. Det franska befälet gick slutligen med på att ge svenskarna fritt avtåg. Fransmänn', 'area': 'Stralsund', 'source_url': 'https://sv.wikipedia.org/wiki/Sveriges_militärhistoria'}
Row 2: {'year': '500', 'title': 'Krig/konflikt i Sverige 500', 'description': 'Göteborgs försvar rustades nu upp i snabb takt, och man utökade exempelvis Göteborgs borgerskaps militärkårer med ytterligare artilleri. Förstärkningstrupper anlände också från flera håll, så att staden snart befann sig i ganska gott försvarsskic

In [28]:
import pandas as pd

# wrap the list of dicts in a DataFrame before using DataFrame methods
df = pd.DataFrame(data)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2626 entries, 0 to 2625
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   year         2626 non-null   object
 1   title        2626 non-null   object
 2   description  2626 non-null   object
 3   area         2626 non-null   object
 4   source_url   2626 non-null   object
dtypes: object(5)
memory usage: 102.7+ KB


In [29]:
def show_counts(df):
    print(df['area'].value_counts())

show_counts(df[df['area'] == "Sverige"])

area
Sverige    65
Name: count, dtype: int64


In [30]:
def get_one_sweden(df):
    print(df[df['area'] == "Sverige"][['year', 'title', 'description', 'area', 'source_url']].iloc[0:6])
    antal_sverige_i_area = df[df['area'] == "Sverige"].count()['area']
    print(f"Antal rader där area är Sverige: {antal_sverige_i_area}")
get_one_sweden(df)

     year               title  \
237  1100  Krig/konflikt 1100   
274  1200  Krig/konflikt 1200   
406  1300  Krig/konflikt 1300   
440  1349  Krig/konflikt 1349   
441  1350  Krig/konflikt 1350   
532  1430  Krig/konflikt 1430   

                                           description     area  \
237  I Sverige inträffade kristendomens slutliga se...  Sverige   
274  I Sverige inträffade kristendomens slutliga se...  Sverige   
406  Senmedeltiden (perioden efter 1350) känneteckn...  Sverige   
440  Senmedeltiden (perioden efter 1350) känneteckn...  Sverige   
441  Senmedeltiden (perioden efter 1350) känneteckn...  Sverige   
532  Kronans ekonomiska svårigheter ledde även till...  Sverige   

                                            source_url  
237  https://www.so-rummet.se/fakta-artiklar/sverig...  
274  https://www.so-rummet.se/fakta-artiklar/sverig...  
406  https://www.so-rummet.se/fakta-artiklar/sverig...  
440  https://www.so-rummet.se/fakta-artiklar/sverig...  
441  https://

In [ ]:
api = ""

In [32]:
from groq import Groq

In [33]:
client = Groq(api_key=api)

In [34]:
def ask_groq_rows(df):
    subset = df[df['area'] == "Sverige"]
    antal_sverige_i_df = subset.count()['area']
    print(f"Antal rader där area är Sverige i hela DataFrame: {antal_sverige_i_df}")
    for index, row in subset.iterrows():
        completion = client.chat.completions.create(
            model="openai/gpt-oss-120b",
            messages=[{
                "role": "system", 
                "content": "Du är en expert på svensk geografi. Svara ENDAST med namnet på den svenska orten eller staden. Om platsen är utanför Sverige, svara 'Utländsk'."
            },
            {
                "role": "user", 
                "content": f"Vilken svensk stad/plats är viktigast här? Svara med max två ord: {row['description']}"
            }]
        )
        print(f"Rad {index}: {completion.choices[0].message.content.strip()}")

ask_groq_rows(df)

Antal rader där area är Sverige i hela DataFrame: 65
Rad 237: Uppsala
Rad 274: Uppsala
Rad 406: Visby
Rad 440: Visby
Rad 441: Visby
Rad 532: Bergslagen
Rad 545: Västerås
Rad 615: Dalarna
Rad 618: Kalmar
Rad 623: Kalmar
Rad 624: Kalmar
Rad 626: Kalmar
Rad 632: Kalmar
Rad 736: Stockholm
Rad 1076: Stockholm
Rad 1197: Älvsborg
Rad 1205: Älvsborg
Rad 1227: Nyköping
Rad 1248: Göteborg
Rad 1255: Älvsborg
Rad 1272: Älvsborg
Rad 1280: Älvsborg
Rad 1309: Stockholm
Rad 1369: Stockholm
Rad 1469: Karlskrona
Rad 1825: Kronslott
Rad 1839: Karlskrona
Rad 1843: Stockholm
Rad 1868: Stockholm
Rad 2014: Stockholm
Rad 2021: Stockholm
Rad 2024: Stockholm
Rad 2032: Stockholm
Rad 2034: Stockholm
Rad 2035: Stockholm
Rad 2036: Stockholm
Rad 2080: Stockholm
Rad 2081: Stockholm
Rad 2148: Stockholm
Rad 2161: Stockholm
Rad 2166: Stockholm
Rad 2172: Stockholm
Rad 2196: Norrköping
Rad 2247: Kungsträdgården
Rad 2248: Stockholm
Rad 2314: Stockholm
Rad 2321: Stockholm
Rad 2338: Stockholm
Rad 2368: Stockholm
Rad 2370: St

In [35]:
def update_area_with_groq(df):
    subset = df[df['area'] == "Sverige"]
    
    for index, row in subset.iterrows():
        completion = client.chat.completions.create(
            model="openai/gpt-oss-120b",
            messages=[
                {"role": "system", "content": "Svara ENDAST med ett svenskt ortnamn eller 'Utländsk'. Svara aldrig 'Okänd', förklara aldrig, och använd inga tecken förutom själva namnet."},
                {"role": "user", "content": f"Ge mig orten i max två ord: {row['description']}"}
            ]
        )
        df.at[index, 'area'] = completion.choices[0].message.content.strip()

update_area_with_groq(df)

In [44]:
def check_replacement(df):
    # Räknar hur många rader som fortfarande har det generiska värdet "Sverige"
    remaining = len(df[df['area'] == "Sverige"])
    print(f"Antal rader kvar med 'Sverige': {remaining}")
    remainingokänd = len(df[df['area'] == "Okänd"])
    print(f"Antal rader kvar med 'Okänd': {remainingokänd}")
    
    # Visar de 10 första raderna för att se de nya ortnamnen
    print("\nFörsta 10 raderna i 'area' kolumnen:")
    print(df['area'].head(10000))

check_replacement(df)

Antal rader kvar med 'Sverige': 0
Antal rader kvar med 'Okänd': 252

Första 10 raderna i 'area' kolumnen:
0       Stralsund
1        Göteborg
2           Visby
3           Skåne
4            Vara
          ...    
2621    Gestilren
2622        Malmö
2623        Polen
2624        Okänd
2625        Malmö
Name: area, Length: 2626, dtype: object


In [45]:
def show_okanda(df):
    # Filtrerar rader där area är "Okänd" och visar de 10 första
    print(df[df['area'] == "Okänd"].head(10))

show_okanda(df)

     year                         title  \
218  1041  Krig/konflikt i Sverige 1041   
244  1100  Krig/konflikt i Sverige 1100   
278  1200  Krig/konflikt i Sverige 1200   
290  1200            Krig/konflikt 1200   
313  1235  Krig/konflikt i Sverige 1235   
314  1235  Krig/konflikt i Sverige 1235   
315  1237  Krig/konflikt i Sverige 1237   
322  1240  Krig/konflikt i Sverige 1240   
323  1241  Krig/konflikt i Sverige 1241   
335  1250  Krig/konflikt i Sverige 1250   

                                           description   area  \
218  Ingvar den vittfarne, född omkring 1016, död 1...  Okänd   
244  Merparten av skandinaverna under vikingatiden ...  Okänd   
278  Merparten av skandinaverna under vikingatiden ...  Okänd   
290  Även om Sverige också under Folkungaätten skak...  Okänd   
313  Birger Magnusson gifte sig omkring 1235–1237, ...  Okänd   
314  Påven Gregorius IX utfärdade 1235 ett dekret m...  Okänd   
315  Birger Magnusson gifte sig omkring 1235–1237, ...  Okänd   
322  M

In [ ]:
def remove_ai_duplicates(df):
    # Filtrera endast de som är "Okänd"
    okand = df[df['area'] == "Okänd"]
    to_drop = []
    
    indices = okand.index.tolist()
    
    for i in range(len(indices)):
        if indices[i] in to_drop: continue
        
        for j in range(i + 1, len(indices)):
            if indices[j] in to_drop: continue
            
            desc1 = df.at[indices[i], 'description']
            desc2 = df.at[indices[j], 'description']
            
            prompt = f"Är dessa två texter samma händelse? Svara bara JA eller NEJ.\n1: {desc1}\n2: {desc2}"
            
            res = client.chat.completions.create(
                model="llama-3.1-8b-instant",
                messages=[{"role": "user", "content": prompt}]
            ).choices[0].message.content.strip().upper()
            
            if "JA" in res:
                to_drop.append(indices[j])
                print(f"Tar bort dubblett: Index {indices[j]}")

    df.drop(to_drop, inplace=True)
    print(f"Klart! Tog bort {len(to_drop)} rader.")

remove_ai_duplicates(df)

Tar bort dubblett: Index 2033
Tar bort dubblett: Index 2119
Tar bort dubblett: Index 2559
